In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from benchmark.config import (
    DATA_PROCESSED,
    DATASETS,
    MODELS_DIR,
    N_TRIALS,
    RESULTS_BENCHMARK,
)
from benchmark.models.random_forest_model import fit_random_forest_cross_validation
from benchmark.models.random_forest_model import fit_best_random_forest

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "diamonds"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name

train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as f:
    metadata = json.load(f)
print(metadata)

{'num_cols': ['carat', 'depth', 'table', 'x', 'y', 'z', 'volumn'], 'cat_cols': ['cut', 'color', 'clarity'], 'target_col': 'price'}


In [3]:
target_column = metadata["target_col"]
x_train = train_df.drop(columns=[target_column]).copy()
y_train = train_df[target_column].to_numpy(dtype=np.float32)

x_test = test_df.drop(columns=[target_column]).copy()
y_test = test_df[target_column].to_numpy(dtype=np.float32)

In [4]:
num_cols = metadata["num_cols"]
cat_cols = metadata["cat_cols"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",
)

In [5]:
best_trial = fit_random_forest_cross_validation(
    X=x_train,
    y=y_train,
    processor=preprocessor,
    task=cfg["task"],
    n_trials=N_TRIALS,
)

rf_model, rf_pipeline, rf_fit_time = fit_best_random_forest(
    X=x_train,
    y=y_train,
    best_trial=best_trial,
    processor=preprocessor,
    task=cfg["task"],
)

[I 2026-04-20 19:08:12,947] A new study created in memory with name: no-name-93abadd2-89c0-4329-9f44-dc0d13bd003b
[I 2026-04-20 19:08:13,501] Trial 0 finished with value: 832770.337919387 and parameters: {'n_estimators': 38, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 832770.337919387.
[I 2026-04-20 19:08:14,416] Trial 1 finished with value: 915247.1249231731 and parameters: {'n_estimators': 93, 'max_depth': 9, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 832770.337919387.
[I 2026-04-20 19:08:15,063] Trial 2 finished with value: 911450.4830338366 and parameters: {'n_estimators': 58, 'max_depth': 9, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 832770.337919387.
[I 2026-04-20 19:08:15,542] Trial 3 finished with value: 1515177.9225547446 and parameter

In [6]:
import time
from benchmark.metrics import evaluate_regression


x_test_transformed = rf_pipeline.transform(x_test)

t0 = time.perf_counter()
y_test_pred = rf_model.predict(x_test_transformed)
inference_time = time.perf_counter() - t0

inference_time_per_sample = inference_time / len(x_test)

test_metrics = evaluate_regression(y_test, y_test_pred)


print("Best params:", best_trial.params)
print("Training time (s):", rf_fit_time)
print("Inference time per sample (s):", inference_time_per_sample)
print("Test metrics:", test_metrics)

Best params: {'n_estimators': 65, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': True}
Training time (s): 0.10165496199988411
Inference time per sample (s): 2.6477425843096e-06
Test metrics: {'rmse': 833.2694}


In [7]:
models_dir = MODELS_DIR / data_name
models_dir.mkdir(parents=True, exist_ok=True)

results_dir = RESULTS_BENCHMARK / data_name
results_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, models_dir / "random_forest.joblib")
joblib.dump(rf_pipeline, models_dir / "random_forest_pipeline.joblib")

output = {
    **test_metrics,
    "fit_time_s": float(rf_fit_time),
    "infer_time_per_sample_s": float(inference_time_per_sample),
    "best_params": best_trial.params,
}

with open(results_dir / "random_forest.json", "w") as f:
    json.dump(output, f, indent=4)

print(results_dir / "random_forest.json")

/home/shado/college_project/comp9417/9417GroupProject/results/diamonds/random_forest.json
